In [2]:
import os, json, random, time
import numpy as np
import cv2
import tensorflow.lite as tflite

# ============================================================
#  BUILD FACE DB (EMBEDDINGS) + THRESHOLD TUNING (OPTION B)
#  - Enroll ALL known images from DATASET_KNOWN_DIR
#  - Use ALL unknown images from UNKNOWN_DIR for threshold tuning
#  - Progress printing with percentages
# ============================================================

# ================== CONFIG ==================
MODEL_PATH = "models\\facenet.tflite"

# Known dataset: ONLY folders of real people (no "unknown" here)
DATASET_KNOWN_DIR = "models\\dataset"

# Unknown dataset: relocated folder containing random people images (flat folder of images)
UNKNOWN_DIR = "unknown"  # <-- put your real path if different (e.g. r"models\unknown")

DB_OUT = "face_db.npz"
THRESH_OUT = "threshold.json"
REPORT_OUT = "db_report.txt"

# Use ALL images:
MAX_IMGS_PER_PERSON = None     # None = use all images
MAX_UNKNOWN_IMGS = None        # None = use all images
RANDOM_SEED = 42               # only used if you keep shuffling

DEFAULT_THRESHOLD = 0.65
UNKNOWN_REJECT_QUANTILE = 0.99

CASCADE_PATH = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
DETECT_DOWNSCALE_MAX_W = 640

PRINT_EVERY = 50               # printing too often slows things; 50 is smoother
# ============================================


# ----------------------------
# Helpers
# ----------------------------
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

face_cascade = cv2.CascadeClassifier(CASCADE_PATH)

def list_images(folder):
    exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [f for f in os.listdir(folder) if f.lower().endswith(exts)]

def l2_normalize(v, eps=1e-10):
    return v / (np.linalg.norm(v) + eps)

def print_progress(prefix, i, total, extra=""):
    if total <= 0:
        return
    pct = (i / total) * 100.0
    msg = f"\r{prefix}: {i}/{total} ({pct:5.1f}%) {extra}"
    print(msg, end="", flush=True)
    if i == total:
        print()

def detect_biggest_face(bgr):
    h, w = bgr.shape[:2]
    scale = 1.0
    if w > DETECT_DOWNSCALE_MAX_W:
        scale = DETECT_DOWNSCALE_MAX_W / w
        bgr_small = cv2.resize(bgr, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    else:
        bgr_small = bgr

    gray = cv2.cvtColor(bgr_small, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.2, minNeighbors=5)
    if len(faces) == 0:
        return None

    x, y, fw, fh = max(faces, key=lambda r: r[2]*r[3])

    if scale != 1.0:
        inv = 1.0 / scale
        x, y, fw, fh = int(x*inv), int(y*inv), int(fw*inv), int(fh*inv)

    pad = int(0.20 * fw)
    x1 = max(0, x - pad); y1 = max(0, y - pad)
    x2 = min(bgr.shape[1], x + fw + pad)
    y2 = min(bgr.shape[0], y + fh + pad)

    face = bgr[y1:y2, x1:x2]
    if face.size == 0:
        return None
    return face


# ----------------------------
# Sanity checks
# ----------------------------
print("Working dir:", os.getcwd())
print("MODEL_PATH:", MODEL_PATH)
print("KNOWN DIR :", DATASET_KNOWN_DIR)
print("UNKNOWN DIR:", UNKNOWN_DIR)

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Missing model file: {MODEL_PATH}")

if not os.path.isdir(DATASET_KNOWN_DIR):
    raise FileNotFoundError(f"Missing known dataset folder: {DATASET_KNOWN_DIR}")


# ----------------------------
# Load TFLite model
# ----------------------------
itp = tflite.Interpreter(model_path=MODEL_PATH, num_threads=4)
itp.allocate_tensors()
inp = itp.get_input_details()[0]
out = itp.get_output_details()[0]

H, W = int(inp["shape"][1]), int(inp["shape"][2])
print("Model input:", inp["shape"], inp["dtype"], "output:", out["shape"], out["dtype"])

def preprocess(face_bgr):
    rgb = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (W, H), interpolation=cv2.INTER_LINEAR).astype(np.float32)
    x = (rgb - 127.5) / 128.0
    return np.expand_dims(x, axis=0).astype(np.float32)

def embed(face_bgr):
    x = preprocess(face_bgr)
    itp.set_tensor(inp["index"], x)
    itp.invoke()
    e = itp.get_tensor(out["index"])[0].astype(np.float32)
    return l2_normalize(e)


# ----------------------------
# Build DB from known people (ALL images)
# ----------------------------
people_known = sorted([
    d for d in os.listdir(DATASET_KNOWN_DIR)
    if os.path.isdir(os.path.join(DATASET_KNOWN_DIR, d))
])

if len(people_known) == 0:
    raise RuntimeError("No person folders found in DATASET_KNOWN_DIR.")

print("\nKnown identities:", people_known)

db = {}
report = []
total_used = 0
total_skipped = 0

t0 = time.time()

for p_idx, person in enumerate(people_known, start=1):
    person_dir = os.path.join(DATASET_KNOWN_DIR, person)
    imgs = list_images(person_dir)

    # If you still want random order, keep shuffle; otherwise comment it out
    # random.shuffle(imgs)

    if MAX_IMGS_PER_PERSON is not None:
        imgs = imgs[:MAX_IMGS_PER_PERSON]

    embs = []
    skipped = 0

    print(f"\n[{p_idx}/{len(people_known)}] Enrolling {person} ({len(imgs)} images)")

    for i, img in enumerate(imgs, start=1):
        if (i % PRINT_EVERY) == 0 or i == len(imgs):
            print_progress(f"  {person}", i, len(imgs), extra=f"ok={len(embs)} skip={skipped}")

        path = os.path.join(person_dir, img)
        bgr = cv2.imread(path)
        if bgr is None:
            skipped += 1
            continue

        face = detect_biggest_face(bgr)
        if face is None:
            skipped += 1
            continue

        try:
            embs.append(embed(face))
        except Exception:
            skipped += 1
            continue

    total_used += len(embs)
    total_skipped += skipped

    if len(embs) == 0:
        report.append(f"{person}: 0 usable faces (skipped={skipped})")
        print(f"⚠️ {person}: no usable faces")
        continue

    db[person] = np.stack(embs, axis=0)
    report.append(f"{person}: {db[person].shape[0]} embeddings (skipped={skipped})")
    print(f"✅ {person}: {db[person].shape[0]} embeddings saved (skipped={skipped})")


np.savez(DB_OUT, **db)
report.append(f"Saved DB: {DB_OUT}")
report.append(f"TOTAL used embeddings: {total_used}")
report.append(f"TOTAL skipped images: {total_skipped}")

print(f"\n✅ DB saved: {DB_OUT}")
print(f"   Identities enrolled: {list(db.keys())}")


# ----------------------------
# Threshold tuning using UNKNOWN_DIR (ALL unknown images)
# ----------------------------
threshold = DEFAULT_THRESHOLD
unk_best_sims = []

if os.path.isdir(UNKNOWN_DIR):
    unk_imgs = list_images(UNKNOWN_DIR)

    # random.shuffle(unk_imgs)  # optional

    if MAX_UNKNOWN_IMGS is not None:
        unk_imgs = unk_imgs[:MAX_UNKNOWN_IMGS]

    if len(unk_imgs) > 0 and len(db) > 0:
        print(f"\nTuning threshold using unknown images: {len(unk_imgs)} images")
        unk_skipped = 0

        for i, img in enumerate(unk_imgs, start=1):
            if (i % PRINT_EVERY) == 0 or i == len(unk_imgs):
                print_progress("  unknown", i, len(unk_imgs), extra=f"collected={len(unk_best_sims)} skip={unk_skipped}")

            bgr = cv2.imread(os.path.join(UNKNOWN_DIR, img))
            if bgr is None:
                unk_skipped += 1
                continue

            face = detect_biggest_face(bgr)
            if face is None:
                unk_skipped += 1
                continue

            try:
                e = embed(face)
            except Exception:
                unk_skipped += 1
                continue

            best = -1.0
            for person, E in db.items():
                s = float(np.max(E @ e))
                if s > best:
                    best = s
            unk_best_sims.append(best)

        if len(unk_best_sims) > 0:
            unk_best_sims = np.array(unk_best_sims, dtype=np.float32)
            threshold = float(np.quantile(unk_best_sims, UNKNOWN_REJECT_QUANTILE))
            report.append(f"Unknown tuning used. best-sim mean={unk_best_sims.mean():.3f}, max={unk_best_sims.max():.3f}")
            report.append(f"Auto threshold (quantile={UNKNOWN_REJECT_QUANTILE}): {threshold:.3f}")
            report.append(f"Unknown skipped images: {unk_skipped}")
            print(f"\n✅ Auto threshold computed: {threshold:.3f}")
            print(f"   Unknown best-sim mean={unk_best_sims.mean():.3f}, max={unk_best_sims.max():.3f}")
        else:
            report.append("Unknown folder exists but no usable unknown faces found. Using default threshold.")
            print("\n⚠️ Unknown folder exists but no usable faces. Using default threshold.")
    else:
        report.append("Unknown folder exists but is empty OR no identities enrolled. Using default threshold.")
        print("\n⚠️ Unknown folder empty OR no identities enrolled. Using default threshold.")
else:
    report.append("Unknown folder not found. Using default threshold.")
    print("\nℹ️ Unknown folder not found. Using default threshold.")


with open(THRESH_OUT, "w") as f:
    json.dump({
        "threshold": float(threshold),
        "model_path": MODEL_PATH,
        "known_dir": DATASET_KNOWN_DIR,
        "unknown_dir": UNKNOWN_DIR if os.path.isdir(UNKNOWN_DIR) else None,
        "input_norm": "(rgb-127.5)/128.0",
        "embedding_dim": int(out["shape"][-1]),
        "max_imgs_per_person": MAX_IMGS_PER_PERSON,
        "max_unknown_imgs": MAX_UNKNOWN_IMGS,
        "unknown_reject_quantile": UNKNOWN_REJECT_QUANTILE
    }, f, indent=2)


with open(REPORT_OUT, "w") as f:
    f.write("\n".join(report) + "\n")

elapsed = time.time() - t0
print("\n" + "\n".join(report))
print(f"✅ Saved: {DB_OUT}, {THRESH_OUT}, {REPORT_OUT}")
print(f"⏱️ Done in {elapsed/60:.1f} minutes")


Working dir: c:\Users\arfao\Desktop\embeded_ai_project
MODEL_PATH: models\facenet.tflite
KNOWN DIR : models\dataset
UNKNOWN DIR: unknown
Model input: [  1 160 160   3] <class 'numpy.float32'> output: [  1 512] <class 'numpy.float32'>

Known identities: ['barhoumi_rami', 'zkaria_arfaoui']

[1/2] Enrolling barhoumi_rami (7933 images)
  barhoumi_rami: 7933/7933 (100.0%) ok=7459 skip=473
✅ barhoumi_rami: 7460 embeddings saved (skipped=473)

[2/2] Enrolling zkaria_arfaoui (8197 images)
  zkaria_arfaoui: 8197/8197 (100.0%) ok=6500 skip=1696
✅ zkaria_arfaoui: 6501 embeddings saved (skipped=1696)

✅ DB saved: face_db.npz
   Identities enrolled: ['barhoumi_rami', 'zkaria_arfaoui']

Tuning threshold using unknown images: 8473 images
  unknown: 8473/8473 (100.0%) collected=6831 skip=1641

✅ Auto threshold computed: 0.745
   Unknown best-sim mean=0.369, max=0.946

barhoumi_rami: 7460 embeddings (skipped=473)
zkaria_arfaoui: 6501 embeddings (skipped=1696)
Saved DB: face_db.npz
TOTAL used embeddings

In [3]:
import numpy as np
import json

DB_IN = "face_db.npz"
DB_OUT = "face_db_compact.npz"
THRESH_IN = "threshold.json"
THRESH_OUT = "threshold_compact.json"

def l2_normalize(v, eps=1e-10):
    return v / (np.linalg.norm(v) + eps)

db = np.load(DB_IN, allow_pickle=True)

compact = {}
for name in db.files:
    E = db[name].astype(np.float32)  # (N,512)
    c = E.mean(axis=0)
    c = l2_normalize(c)
    compact[name] = c[None, :]       # shape (1,512)

np.savez(DB_OUT, **compact)
print("✅ Saved compact DB:", DB_OUT)

# copy threshold for now (we'll re-tune next)
with open(THRESH_IN, "r") as f:
    th = json.load(f)
with open(THRESH_OUT, "w") as f:
    json.dump(th, f, indent=2)

print("✅ Copied threshold to:", THRESH_OUT)


✅ Saved compact DB: face_db_compact.npz
✅ Copied threshold to: threshold_compact.json
